# 第3日 ノートブック01：YOLO を理解する

# Day 3 Notebook 01: Understanding YOLO

<!-- ne:begin after-heading -->
# दिन ३ नोटबुक ०१: YOLO बुझ्ने
<!-- ne:end -->

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 物体検出モデルが何をするのか説明できる | 1. Explain what an object detection model does |
| 2. バウンディングボックスと座標の表し方を説明できる | 2. Explain bounding boxes and their coordinates |
| 3. 確からしさ（confidence）の意味を説明できる | 3. Explain what confidence means |
| 4. IoU を手計算できる | 4. Compute IoU by hand |
| 5. 事前学習モデルが塩ビ欠陥を検出できない理由を説明できる | 5. Explain why a pretrained model cannot detect PVC defects |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 80 分 / 80 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第1日・第2日を終えていること。 / You have finished Days 1 and 2. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| IoU の手計算結果 | Your hand-computed IoU |
| YOLO 形式の座標計算 | A YOLO-format coordinate calculation |

<!-- ne:begin end -->
## सिकाइ लक्ष्यहरू

| नेपाली |
| --- |
| 1. वस्तु पहिचान गर्ने model ले के गर्छ बताउन सक्ने। |
| 2. बाउन्डिङ बक्स र निर्देशाङ्क लेख्ने तरिका बताउन सक्ने। |
| 3. विश्वास स्कोर (confidence) को अर्थ बताउन सक्ने। |
| 4. IoU हातैले गणना गर्न सक्ने। |
| 5. पहिले नै तालिम भएको model ले PVC को दोष किन भेट्न नसक्छ बताउन सक्ने। |

## यो नोटबुकको आधारभूत जानकारी

| विषय | विवरण |
| --- | --- |
| सिकाइ समय (अनुमान) | 80 मिनेट |
| कठिनाइ | सुरुवाती तह |
| पूर्वशर्त | दिन १ र दिन २ सकिएको हुनुपर्छ। |
| आवश्यक लाइब्रेरी | numpy, matplotlib, pillow |
| प्रयोग हुने kernel | aivision |
| आजको समयतालिका | [`COURSE_SYLLABUS.md` दिन ३](../../COURSE_SYLLABUS.md#day3) |

### बुझाउनुपर्ने काम

| नेपाली |
| --- |
| IoU हातैले गणना गरेको नतिजा |
| YOLO ढाँचाको निर्देशाङ्क गणना |
<!-- ne:end -->


### confidence と正しさは別 / confidence is not correctness

<!-- ne:begin after-heading -->
### confidence र सही हुनु फरक कुरा हुन्
<!-- ne:end -->

![](../../assets/diagrams/teach_score_vs_correctness.png)

図：5 つの検出の confidence と、正解と比べた正しさ（値は例示用）。

Figure: confidence against correctness for five detections (illustrative values).

**気づいてほしいこと**

confidence が高い検出でも間違っていることがあり、低い検出でも正しいことがあります。confidence はモデルが出したスコアで、正解でも「正しい確率」でもありません。正しさは正解ラベルと対応づけのルールで確かめます。

**What to notice**

A high-confidence detection can be wrong and a low-confidence one can be right. Confidence is a score produced by the model: it is neither ground truth nor a probability of being correct. Correctness is checked against ground truth with a stated matching rule.

**考えてみてください / Try to answer**

- しきい値 0.60 で残る検出のうち、正しくないのはどれですか。
- At a threshold of 0.60, which of the surviving detections are wrong?

<!-- ne:begin end -->
चित्र: 5 वटा पहिचानको confidence र सही उत्तरसँग तुलना गर्दा कति सही भयो (मान उदाहरणका लागि मात्र)।

**ध्यान दिनुपर्ने कुरा**

confidence बढी भएको पहिचान पनि गलत हुन सक्छ, र कम भएको पनि सही हुन सक्छ। confidence भनेको model
ले दिएको स्कोर हो; यो सही उत्तर पनि होइन, "सही हुने सम्भावना" पनि होइन। सही छ कि छैन भन्ने कुरा सही
लेबलसँग तोकिएको नियमअनुसार मिलाएर जाँचिन्छ।

**सोच्नुहोस्**

- थ्रेसहोल्ड 0.60 मा बाँकी रहने पहिचानमध्ये कुन-कुन गलत छन्?
<!-- ne:end -->


### 予想して、実行して、説明する / Predict, run, inspect, explain

<!-- ne:begin after-heading -->
### अनुमान गर्नुहोस्, चलाउनुहोस्, हेर्नुहोस्, व्याख्या गर्नुहोस्
<!-- ne:end -->

クラスは欠陥の種類です。confidence はモデルのスコアで、正解の保証ではありません。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。x2 だけを変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

A class is a defect category; confidence is a model score, not guaranteed correctness. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only x2, predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.

<!-- ne:begin end -->
class भनेको दोषको प्रकार हो। confidence model को स्कोर हो, सही हुने ग्यारेन्टी होइन। महत्त्वपूर्ण सेल
चलाउनुअघि इनपुट, काम र नतिजा एक-एक वाक्यमा लेख्नुहोस्। पहिले अनुमान गरेर चलाउनुहोस् र मान वा चित्र
हेर्नुहोस्। `x2` मात्र बदलेर फेरि अनुमान गरी चलाउनुहोस्। जानाजान राखिएका त्रुटि र नसकिएका अभ्यासका
निर्देशन पछ्याउनुहोस्।
<!-- ne:end -->


## このノートブックの位置 / Where this notebook sits

<!-- ne:begin after-heading -->
## यो नोटबुक कहाँ पर्छ
<!-- ne:end -->

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第3日 / Day 3 |
| この日の順番 / Position today | 1 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 80 分 / 80 minutes |
| 前提 / Prerequisites | 第2日 / Day 2 |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.

<!-- ne:begin end -->
| विषय | विवरण |
| --- | --- |
| दिन | दिन ३ |
| आजको क्रम | आजको नोटबुकमध्ये 1 / 3 |
| लाग्ने समय | 80 मिनेट |
| पूर्वशर्त | दिन २ |
| आजको समयतालिका | [`COURSE_SYLLABUS.md` दिन ३](../../COURSE_SYLLABUS.md#day3) |

दिनको समयतालिका `COURSE_SYLLABUS.md` मा एउटै ठाउँमा छ। यो नोटबुक त्यसको एउटा भाग मात्र हो।
<!-- ne:end -->


---

## Part 0　準備 / Preparation

<!-- ne:begin after-heading -->
## Part 0 तयारी
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

#### YOLOとは / What YOLO does

<!-- ne:begin after-heading -->
#### YOLO के हो
<!-- ne:end -->

YOLO（You Only Look Once）は画像から四角・学習したクラス・スコアを予測する物体検出モデルです。
モデルの順伝播に後処理を加えて結果を作ります。「1回の計算だけ」ではありません。
ライン速度に間に合うかは、実機・画像サイズ・前処理・後処理を含めて測定しなければ分かりません。

YOLO (You Only Look Once) is an object detector predicting boxes, learned classes and scores.
A model forward pass plus postprocessing produces the result; it is not literally “one calculation”.
Production throughput requires benchmarking on the target hardware, including input size,
preprocessing and postprocessing.

<!-- ne:begin end -->
YOLO (You Only Look Once) भनेको तस्बिरबाट आयत, तालिम लिएका class र स्कोर अनुमान गर्ने वस्तु-पहिचान
model हो। model को forward pass र postprocessing मिलेर नतिजा बन्छ; "एकै पटकको गणना मात्र" भन्ने होइन।
उत्पादन लाइनको गतिमा समय पुग्छ कि पुग्दैन भन्ने कुरा वास्तविक यन्त्र, तस्बिरको आकार, पूर्व-प्रशोधन र
पछि-प्रशोधन सहित नापेर मात्र थाहा हुन्छ।
<!-- ne:end -->


---

## Part 1　バウンディングボックス / Bounding boxes

<!-- ne:begin after-heading -->
## Part 1 बाउन्डिङ बक्स
<!-- ne:end -->


### 四角を画像の上で読む / reading a box on the image

<!-- ne:begin after-heading -->
### तस्बिरमा आयत पढ्ने
<!-- ne:end -->

![](../../assets/diagrams/teach_bbox_coords.png)

図：同じ 1 つの欠陥の四角を、画素の数（左）と画像サイズで割った値（右）で表したもの。座標は計算して図に描いています。

Figure: one defect box written as pixel counts (left) and as fractions of the image size (right). The coordinates are computed and drawn, not sketched.

**気づいてほしいこと**

左は画素の xyxy、右は画像サイズで割った cx, cy, w, h で、**同じ箱**です。画像全体の縮小・拡大では 0〜1 の値は意味を保ちますが、切り抜き（crop）や余白の追加（letterbox）では座標が変わります。API ごとに形式が違うので名前を確認します（Ultralytics の `xyxy`、OpenCV の `boundingRect` は xywh）。

**What to notice**

The left is pixel xyxy; the right is cx, cy, w, h divided by the image size — **the same box**. Resizing the whole image keeps the 0–1 values meaningful, but cropping or letterbox padding changes the coordinates. Formats differ by API (Ultralytics `xyxy`; OpenCV `boundingRect` is xywh).

**考えてみてください / Try to answer**

- 画像の左半分だけを切り抜いたとき、もとの cx = 0.6 の箱はどうなりますか。
- If you crop the left half of the image, what happens to a box with cx = 0.6?

<!-- ne:begin end -->
चित्र: एउटै दोषको आयतलाई बायाँ पिक्सेल सङ्ख्याले र दायाँ तस्बिरको आकारले भाग गरेको मानले देखाइएको छ।
निर्देशाङ्क गणना गरेर चित्रमा राखिएका हुन्।

**ध्यान दिनुपर्ने कुरा**

बायाँ पिक्सेलको xyxy र दायाँ तस्बिरको आकारले भाग गरेको cx, cy, w, h — **एउटै बक्स** हो। पूरै तस्बिर
सानो-ठुलो गर्दा 0–1 का मान अर्थपूर्ण रहन्छन्, तर काट्दा (crop) वा खाली ठाउँ थप्दा (letterbox) निर्देशाङ्क
बदलिन्छन्। API-अनुसार ढाँचा फरक हुन्छ, नाम जाँच्नुहोस् (Ultralytics को `xyxy`, OpenCV को
`boundingRect` xywh हो)।

**सोच्नुहोस्**

- तस्बिरको बायाँ आधा भाग मात्र काट्दा, पहिले cx = 0.6 भएको बक्स के हुन्छ?
<!-- ne:end -->


#### 四角で囲む、4 つの数値

<!-- ne:begin after-heading -->
#### आयतले घेर्ने, चार सङ्ख्या
<!-- ne:end -->
バウンディングボックス（以下「ボックス」）は、欠陥を囲む四角です。
ボックスは 4 つの数値で表し、並べ方（**座標の形式**）は API ごとに決まっています。

| 形式 | 4 つの数値 | 使う場所 |
|---|---|---|
| `xyxy` | `x1, y1, x2, y2`（左上と右下） | Ultralytics の `result.boxes.xyxy`、描画ライブラリ |
| `xywh` | `x, y, w, h`（左上と幅・高さ） | `cv2.boundingRect()`（OpenCV の輪郭の外接矩形） |
| `xc, yc, w, h`（正規化） | 中心と幅・高さを画像サイズで割った 0〜1 の値 | YOLO のラベルファイル（正解データ） |

同じ「4 つの数値」でも、`xyxy`・`xywh`・YOLO のラベルは **意味も順番も違います**。
関数を呼ぶたびに、どの形式が返るのかを確認してください。

YOLO のラベルは **中心方式** で、しかも **画像の幅・高さで割った 0〜1 の値**（正規化）です。
この正規化座標は、**画像全体を同じ比率で縮小・拡大する**（全体をリサイズする）ときは同じ値のまま使えます。
一方、画像の一部を切り抜いた（crop）ときや、縦横比を保つために余白を足した（letterbox）ときは、
座標の基準になる枠が変わるので、値はそのままでは使えません。
基準の枠が変わったら、新しい枠の幅・高さで計算し直します。

#### A rectangle is four numbers
A bounding box marks the defect with a rectangle. It is written as four numbers, and the
**order** of those numbers (**the coordinate format**) is fixed by each API.

| Format | Four numbers | Where it is used |
|---|---|---|
| `xyxy` | `x1, y1, x2, y2` (top-left, bottom-right) | Ultralytics `result.boxes.xyxy`, drawing libraries |
| `xywh` | `x, y, w, h` (top-left, width, height) | `cv2.boundingRect()` (the OpenCV contour rectangle) |
| `xc, yc, w, h` (normalised) | centre and size divided by the image size, in 0-1 | YOLO label files (ground truth) |

The same four numbers mean different things in `xyxy`, in `xywh` and in a YOLO label. Check which
format each function returns.

YOLO labels use the **centre style**, normalised **by the image width and height** into 0-1.
These normalised values stay valid when the **whole image** is resized by the same factor.
They do not stay valid after a crop, or after letterbox padding added to keep the aspect ratio,
because the reference frame changes. Recompute the values against the new frame.

<!-- ne:begin end -->
बाउन्डिङ बक्सले दोषलाई आयतले घेर्छ। बक्स चार सङ्ख्याले लेखिन्छ, र ती सङ्ख्याको क्रम
(**निर्देशाङ्कको ढाँचा**) API-अनुसार तोकिएको हुन्छ।

| ढाँचा | चार सङ्ख्या | कहाँ प्रयोग हुन्छ |
|---|---|---|
| `xyxy` | `x1, y1, x2, y2` (माथि-बायाँ र तल-दायाँ) | Ultralytics को `result.boxes.xyxy`, चित्र बनाउने लाइब्रेरी |
| `xywh` | `x, y, w, h` (माथि-बायाँ र चौडाइ, उचाइ) | `cv2.boundingRect()` (OpenCV को कन्टुर वरिपरिको आयत) |
| `xc, yc, w, h` (सामान्यीकृत) | केन्द्र र आकारलाई तस्बिरको आकारले भाग गरेको 0–1 मान | YOLO को label फाइल (सही लेबल) |

उही "चार सङ्ख्या" भए पनि `xyxy`, `xywh` र YOLO label मा **अर्थ र क्रम फरक** हुन्छ। function बोलाउँदा
कुन ढाँचा फर्किन्छ सधैं जाँच्नुहोस्।

YOLO label **केन्द्र-शैली** मा हुन्छ र **तस्बिरको चौडाइ-उचाइले भाग गरेको 0–1 मान** (सामान्यीकृत) हुन्छ।
पूरै तस्बिर एउटै अनुपातमा सानो-ठुलो गर्दा यी मान उही रहन्छन्। तर तस्बिरको भाग काट्दा (crop) वा अनुपात
जोगाउन खाली ठाउँ थप्दा (letterbox) आधार बन्ने फ्रेम बदलिन्छ, त्यसैले त्यसै प्रयोग गर्न मिल्दैन। फ्रेम
बदलियो भने नयाँ फ्रेमको चौडाइ-उचाइले फेरि गणना गर्नुहोस्।
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show([Image.open(ASSETS / "diagrams" / "diagram_yolo_annotation.png")],
     ["YOLO annotation: normalised centre, width, height"], figsize=(9, 7))

#### 座標変換を自分で計算する

<!-- ne:begin after-heading -->
#### निर्देशाङ्क परिवर्तन आफैं गणना गर्ने
<!-- ne:end -->
ピクセル座標 `(x1, y1, x2, y2)` から YOLO 形式への変換は 4 行です。

```text
xc = (x1 + x2) / 2 / 画像の幅
yc = (y1 + y2) / 2 / 画像の高さ
w  = (x2 - x1)     / 画像の幅
h  = (y2 - y1)     / 画像の高さ
```

割るのを忘れると、値が 1 を超えます。**0〜1 の範囲かどうかを必ず確認** してください。

この 4 行が使うのは **元の画像全体の幅と高さ** です。切り抜き（crop）や letterbox の余白を
入れたあとなら、新しい画像の幅・高さで計算し直してください。古い値をそのまま使うと枠がずれます。

#### Do the conversion yourself
Converting pixel corners to YOLO format takes four lines:

```text
xc = (x1 + x2) / 2 / image_width
yc = (y1 + y2) / 2 / image_height
w  = (x2 - x1)     / image_width
h  = (y2 - y1)     / image_height
```

Forgetting the division gives values above 1. **Always check the 0-1 range.**

These four lines divide by the width and height of the **whole original image**. After a crop, or
after letterbox padding, recompute with the new frame size; reusing the old values shifts the box.

<!-- ne:begin end -->
पिक्सेल निर्देशाङ्क `(x1, y1, x2, y2)` बाट YOLO ढाँचामा बदल्न चार लाइन पुग्छ:

```text
xc = (x1 + x2) / 2 / तस्बिरको चौडाइ
yc = (y1 + y2) / 2 / तस्बिरको उचाइ
w  = (x2 - x1)     / तस्बिरको चौडाइ
h  = (y2 - y1)     / तस्बिरको उचाइ
```

भाग गर्न बिर्से मान 1 भन्दा बढी हुन्छ। **0–1 दायराभित्र छ कि छैन सधैं जाँच्नुहोस्।**

यी चार लाइनले **मूल तस्बिरको पूरै चौडाइ र उचाइ** प्रयोग गर्छन्। काटेपछि (crop) वा letterbox को खाली
ठाउँ थपेपछि नयाँ तस्बिरको चौडाइ-उचाइले फेरि गणना गर्नुहोस्। पुरानै मान प्रयोग गर्दा बक्स सर्छ।
<!-- ne:end -->


### コードの読み方 / Reading the code

<!-- ne:begin after-heading -->
### कोड कसरी पढ्ने
<!-- ne:end -->

![](../../assets/svg/teach_bbox_one_example.svg)

図：class_id は整数で、正規化しません。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: class_id is an integer and is not normalised. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

class_id は整数で、正規化しません。 入力と変換後の値を指でたどってください。

**What to notice**

class_id is an integer and is not normalised. Trace the input and transformed values.

**考えてみてください / Try to answer**

x2 だけを24 px増やすと中心と幅はどう変わりますか。

How do centre and width change if only x2 grows by 24 px?

<!-- ne:begin end -->
चित्र: class_id पूर्णाङ्क हो र सामान्यीकृत गरिँदैन। (गणना गरिएको संश्लेषित उदाहरण र योजनाबद्ध चित्र;
model को वास्तविक नापिएको नतिजा होइन।)

**ध्यान दिनुपर्ने कुरा**

class_id पूर्णाङ्क हो र सामान्यीकृत गरिँदैन। इनपुट र परिवर्तन भएको मान औंलाले पछ्याउनुहोस्।

**सोच्नुहोस्**

`x2` मात्र 24 px बढाउँदा केन्द्र र चौडाइ कसरी बदलिन्छ?
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
image_width, image_height = 240, 240
x1, y1, x2, y2 = 96, 72, 168, 132

xc = (x1 + x2) / 2 / image_width
yc = (y1 + y2) / 2 / image_height
w = (x2 - x1) / image_width
h = (y2 - y1) / image_height

print(f"center x : {xc:.3f}")
print(f"center y : {yc:.3f}")
print(f"width    : {w:.3f}")
print(f"height   : {h:.3f}")
print()
print("0〜1 の範囲か / inside 0..1 ?", all(0 <= v <= 1 for v in (xc, yc, w, h)))
print("ラベルファイルの 1 行 / one label line:")
print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

### 実行結果の例 / Expected output

<!-- ne:begin after-heading -->
### नतिजाको उदाहरण
<!-- ne:end -->

この例では次の結果になります（同じ入力なら同じ結果です）。

```text
center x : 0.550
center y : 0.425
width    : 0.300
height   : 0.250

0〜1 の範囲か / inside 0..1 ? True
ラベルファイルの 1 行 / one label line:
0 0.550000 0.425000 0.300000 0.250000
```

This example produces the following result (the same input gives the same result).

```text
center x : 0.550
center y : 0.425
width    : 0.300
height   : 0.250

0〜1 の範囲か / inside 0..1 ? True
ラベルファイルの 1 行 / one label line:
0 0.550000 0.425000 0.300000 0.250000
```


---

#### 同じ四角を3形式で読む / Read the same box in three formats

<!-- ne:begin after-heading -->
#### उही आयत तीन ढाँचामा पढ्ने
<!-- ne:end -->

入力は既出の240×240画像上の `(96,72,168,132)` です。下のセルは3形式の対応表を出します。
予想：右端x2だけを24 px増やすと、幅と中心xは何px増えますか。yの値は変わりますか。

Input is the same `(96,72,168,132)` box on the 240×240 image. The cell prints three representations.
Predict how width and centre x change if only x2 grows by 24 px. Do any y values change?

<!-- ne:begin end -->
इनपुट 240×240 तस्बिरमा पहिले नै देखाइएको `(96,72,168,132)` हो। तलको सेलले तीन ढाँचाको तालिका देखाउँछ।
अनुमान: दायाँ छेउ `x2` मात्र 24 px बढाउँदा चौडाइ र केन्द्र x कति px बढ्छ? y का मान बदलिन्छन्?
<!-- ne:end -->


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
box_image_width, box_image_height = 240, 240
right_edge_change = 24  # ← この値だけ変更 / change only this value
for label, box in [("original", (96, 72, 168, 132)),
                   ("edited", (96, 72, 168 + right_edge_change, 132))]:
    left, top, right, bottom = box
    if not (0 <= left < right <= box_image_width and 0 <= top < bottom <= box_image_height):
        raise ValueError("画像内の正の四角が必要 / box must have positive size and stay inside image")
    box_width, box_height = right - left, bottom - top
    centre_x, centre_y = (left + right) / 2, (top + bottom) / 2
    yolo_box = (centre_x / box_image_width, centre_y / box_image_height,
                box_width / box_image_width, box_height / box_image_height)
    print(label, "xyxy:", box)
    print("OpenCV top-left xywh:", (left, top, box_width, box_height))
    print("YOLO:", "0 " + " ".join(f"{v:.6f}" for v in yolo_box))


幅は72→96、中心xは132→144です。YOLOのxcとwだけが変わり、ycとhは同じです。
24を12へ変えて予想し、元の図と異なる四角であることを説明してください。

Width changes 72→96, centre x 132→144. Only YOLO xc and w change; yc and h stay fixed.
Try 12 instead of 24 and explain why the edited rectangle differs from the original figure.

<!-- ne:begin end -->
चौडाइ 72 बाट 96, केन्द्र x 132 बाट 144 हुन्छ। YOLO को xc र w मात्र बदलिन्छन्; yc र h उही रहन्छन्।
24 को सट्टा 12 राखेर अनुमान गर्नुहोस् र बदलेको आयत मूल चित्रको आयतभन्दा किन फरक छ व्याख्या गर्नुहोस्।
<!-- ne:end -->


## Part 2　クラスと確からしさ / Classes and confidence

<!-- ne:begin after-heading -->
## Part 2 class र विश्वास स्कोर
<!-- ne:end -->


#### クラスラベル

<!-- ne:begin after-heading -->
#### class लेबल
<!-- ne:end -->
**クラス** は「種類」のことです。YOLO は、学習した種類のどれかに分類します。

一般的な事前学習モデル（COCO データセット）は 80 種類を学習しています。
例：`person`, `car`, `bus`, `dog`, `bottle` ……

ここに **`black_spot`（黒点）や `scratch`（傷）はありません**。
だから塩ビの欠陥は検出できません。

#### Class label
A **class** is a category. YOLO assigns each detection to one of its trained classes.

The common pretrained model (COCO dataset) knows 80 classes, for example
`person`, `car`, `bus`, `dog`, `bottle` ...

**`black_spot` and `scratch` are not among them**, which is why it cannot detect PVC defects.

<!-- ne:begin end -->
**class** भनेको "प्रकार" हो। YOLO ले पहिचान तालिम लिएका प्रकारमध्ये एउटामा राख्छ।

सामान्य पहिले नै तालिम भएको model (COCO डेटासेट) ले 80 प्रकार जान्दछ। जस्तै `person`, `car`, `bus`,
`dog`, `bottle` …

यसमा **`black_spot` (कालो थोप्लो) वा `scratch` (चिसो) छैनन्**। त्यसैले PVC को दोष भेट्न सक्दैन।
<!-- ne:end -->


#### 確からしさ（confidence）

<!-- ne:begin after-heading -->
#### विश्वास स्कोर (confidence)
<!-- ne:end -->
**confidence** は、モデルが各候補に付けた **スコア** です。0〜1 の値をとります。

| 値 | 読み方 |
|---|---|
| 0.95 | スコアが高い（モデルが強く支持している） |
| 0.50 | 中間のスコア |
| 0.10 | スコアが低い |

**このスコアは「正しい確率」ではありません。**
0.8 を「80% の確率で正しい」と読むことはできませんし、0.50 も「五分五分の確率で正しい」ではありません。
スコアは、学習したクラスの見た目にどれくらい近いかをモデルが数値にしたものです。
**スコアが高くても正しいとは限らず、低くても正しい検出はあります。**
正しさは、正解ラベルと決められた対応付けのルール（例：IoU がしきい値以上）で確認します。

#### Confidence
**Confidence** is a **score** the model assigns to each candidate, between 0 and 1.

| Value | How to read it |
|---|---|
| 0.95 | high score (the model strongly supports it) |
| 0.50 | middling score |
| 0.10 | low score |

**This score is not a probability of correctness.** You cannot read 0.8 as "80% likely to be
correct", and 0.50 does not mean an even chance of being correct. The score says how closely the
candidate matches the appearance of the class the model learned.
**A high score does not guarantee a correct detection, and a low score does not make a detection
wrong.** Correctness is checked against ground truth with a stated matching rule, for example
IoU at or above a threshold.

<!-- ne:begin end -->
**confidence** model ले प्रत्येक उम्मेदवारलाई दिएको **स्कोर** हो। यो 0–1 बीचको मान हुन्छ।

| मान | कसरी पढ्ने |
|---|---|
| 0.95 | स्कोर उच्च (model ले बलियो समर्थन गरेको) |
| 0.50 | मध्यम स्कोर |
| 0.10 | स्कोर न्यून |

**यो स्कोर "सही हुने सम्भावना" होइन।** 0.8 लाई "80% सम्भावनाले सही" भनेर पढ्न मिल्दैन, 0.50 पनि
"आधा-आधा सम्भावना" होइन। स्कोरले तालिम लिएको class को देखिने रूपसँग कति मिल्छ भन्ने कुरा model ले
सङ्ख्यामा बदलेको हो। **स्कोर उच्च भए पनि सही हुन्छ भन्ने छैन, र कम भए पनि सही पहिचान हुन्छ।** सही छ कि
छैन भन्ने कुरा सही लेबलसँग तोकिएको नियम (जस्तै IoU कुनै थ्रेसहोल्ड वा बढी) ले जाँचिन्छ।
<!-- ne:end -->


---

## Part 3　IoU（重なりの度合い）/ Intersection over Union

<!-- ne:begin after-heading -->
## Part 3 IoU (ओभरल्यापको मात्रा)
<!-- ne:end -->


### 交差・和集合・IoU / intersection, union and IoU

<!-- ne:begin after-heading -->
### प्रतिच्छेद, संयोग र IoU
<!-- ne:end -->

![](../../assets/diagrams/teach_iou_rects.png)

図：A（正解）と B（予測）の四角を実際に重ね、交差（重なりの面積）と和集合（2 つの枠を合わせた領域。重なりは 1 回だけ数える）を計算したもの。この図では 6,750 ÷ 15,150 ≒ 0.45 です。

Figure: truth box A and predicted box B actually overlapped, with the intersection and the union (the combined area of the two boxes, with the overlap counted once) computed. Here 6,750 / 15,150 ≈ 0.45.

**気づいてほしいこと**

IoU は「重なりの度合い」を 1 つの数にしたものです。0 に近いほど重なりが少なく、1 に近いほどぴったり一致しています。**面積の比** なので、単位はありません。和集合は 2 つの面積を足すだけでは求まりません。重なりを 2 回数えないよう、1 回だけ数えます。

**What to notice**

IoU turns overlap into one number. Near 0 means little overlap; near 1 means a tight fit. It is a **ratio of areas**, so it has no unit. The union is not the sum of the two areas: the overlap is counted once, not twice.

**考えてみてください / Try to answer**

- IoU が 0.3 のとき、重なりの面積は「予測の枠」の 30 % ですか、「正解の枠」の 30 % ですか、それとも別のものの 30 % ですか。
- When IoU is 0.3, is the intersection 30 % of the predicted box, 30 % of the truth box, or 30 % of something else?

<!-- ne:begin end -->
चित्र: A (सही उत्तर) र B (अनुमान) का आयत साँच्चै मिलाएर प्रतिच्छेद (ओभरल्यापको क्षेत्रफल) र संयोग (दुई
आयत मिलेर बन्ने क्षेत्र; ओभरल्याप एक पटक मात्र गनिन्छ) गणना गरिएको। यो चित्रमा 6,750 ÷ 15,150 ≒ 0.45 हो।

**ध्यान दिनुपर्ने कुरा**

IoU ले "ओभरल्यापको मात्रा" एउटा सङ्ख्यामा बदल्छ। 0 नजिक हुँदा ओभरल्याप कम, 1 नजिक हुँदा ठ्याक्कै
मिलेको। यो **क्षेत्रफलको अनुपात** हुनाले एकाइ हुँदैन। संयोग दुई क्षेत्रफल जोडेर मात्र पाइँदैन; ओभरल्याप
दुई पटक नगन्न एक पटक मात्र गनिन्छ।

**सोच्नुहोस्**

- IoU 0.3 हुँदा प्रतिच्छेद "अनुमानको आयत" को 30% हो, "सही आयत" को 30% हो, कि अरू कुनै कुराको 30%?
<!-- ne:end -->


#### IoU とは「2 つの四角の重なり具合」

<!-- ne:begin after-heading -->
#### IoU भनेको "दुई आयतको ओभरल्यापको मात्रा"
<!-- ne:end -->
**IoU** は、予測したボックスと正解のボックスが
**どれくらい重なっているか** を 0〜1 で表す数値です。

```text
IoU = 重なりの面積 ÷ 2 つの枠を合わせた領域の面積（重なりは 1 回だけ数える）
```

「2 つの枠を合わせた領域」が **和集合** です。2 つの面積をそのまま足すと重なりを 2 回
数えてしまうので、重なりを 1 回だけ数えるように足し引きします。

- `IoU = 1.0` … 完全に一致
- `IoU = 0.5` … まあまあ合っている
- `IoU = 0.0` … まったく重なっていない

`IoU = 0.3` は「重なりの面積が、**2 つの枠を合わせた領域の** 30 %」という意味です。
予測の枠の 30 %、正解の枠の 30 % という意味ではありません。

一般に `IoU >= 0.5` を「正しく検出できた」とみなすことが多いです。
ただし、これは **慣習** であり、用途によって変わります。

#### IoU measures how much two boxes overlap
**IoU** expresses how much a predicted box overlaps the true box, between 0 and 1.

```text
IoU = area of the intersection / area of the union (count the overlap once)
```

The union is the combined region of the two boxes. Adding the two areas directly would count the
overlap twice, so the subtraction counts it once.

- `IoU = 1.0` identical
- `IoU = 0.5` roughly matching
- `IoU = 0.0` no overlap

`IoU = 0.3` means the intersection is 30 % of the **union of the two boxes**. It does not mean 30 %
of the predicted box, nor 30 % of the true box.

A detection is often counted as correct when `IoU >= 0.5`, though that is a **convention** that
depends on the application.

<!-- ne:begin end -->
**IoU** ले अनुमान गरिएको बक्स र सही बक्स **कति ओभरल्याप छन्** भन्ने कुरा 0–1 मा देखाउँछ।

```text
IoU = प्रतिच्छेदको क्षेत्रफल ÷ दुई आयत मिलेर बन्ने क्षेत्रफल (ओभरल्याप एक पटक मात्र)
```

"दुई आयत मिलेर बन्ने क्षेत्र" नै **संयोग** हो। दुई क्षेत्रफल त्यसै जोड्दा ओभरल्याप दुई पटक गनिन्छ,
त्यसैले ओभरल्याप एक पटक मात्र गनिने गरी जोडघटाउ गरिन्छ।

- `IoU = 1.0` … ठ्याक्कै मिल्यो
- `IoU = 0.5` … ठिकै मिल्यो
- `IoU = 0.0` … केही पनि ओभरल्याप छैन

`IoU = 0.3` को अर्थ "प्रतिच्छेदको क्षेत्रफल **दुई आयत मिलेर बन्ने क्षेत्रको** 30%" हो। अनुमानको आयतको
30% वा सही आयतको 30% होइन।

सामान्यतया `IoU >= 0.5` लाई "सही पहिचान" मानिन्छ। तर यो **चलन** हो, प्रयोगअनुसार फरक पर्छ।
<!-- ne:end -->


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
def iou(box_a, box_b):
    """2 つのボックスの IoU を計算する / compute IoU of two (x1, y1, x2, y2) boxes."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # 重なっている部分 / the intersection rectangle
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


truth = (100, 100, 200, 200)          # 100 x 100 = 10,000 px2
cases = [
    ("完全一致 / identical      ", (100, 100, 200, 200)),
    ("少しずれる / shifted      ", (110, 110, 210, 210)),
    ("半分ずれる / half overlap ", (150, 100, 250, 200)),
    ("まったく別 / disjoint     ", (300, 300, 400, 400)),
]

print("正解 / truth :", truth)
print()
for label, box in cases:
    print(f"{label}  {box}  ->  IoU = {iou(truth, box):.3f}")

### 実行結果の例 / Expected output

<!-- ne:begin after-heading -->
### नतिजाको उदाहरण
<!-- ne:end -->

この例では次の結果になります（同じ入力なら同じ結果です）。

```text
正解 / truth : (100, 100, 200, 200)

完全一致 / identical        (100, 100, 200, 200)  ->  IoU = 1.000
少しずれる / shifted        (110, 110, 210, 210)  ->  IoU = 0.681
半分ずれる / half overlap   (150, 100, 250, 200)  ->  IoU = 0.333
まったく別 / disjoint       (300, 300, 400, 400)  ->  IoU = 0.000
```

This example produces the following result (the same input gives the same result).

```text
正解 / truth : (100, 100, 200, 200)

完全一致 / identical        (100, 100, 200, 200)  ->  IoU = 1.000
少しずれる / shifted        (110, 110, 210, 210)  ->  IoU = 0.681
半分ずれる / half overlap   (150, 100, 250, 200)  ->  IoU = 0.333
まったく別 / disjoint       (300, 300, 400, 400)  ->  IoU = 0.000
```


#### 手計算で確認する

<!-- ne:begin after-heading -->
#### हातैले गणना गरेर जाँच्ने
<!-- ne:end -->
「少しずれる」場合を手で計算してみましょう。

- 正解：`(100,100)` 〜 `(200,200)` → 面積 **10,000**
- 予測：`(110,110)` 〜 `(210,210)` → 面積 **10,000**
- 重なり：`(110,110)` 〜 `(200,200)` → 90 × 90 = **8,100**
- 2 つの枠を合わせた領域（和集合）：10,000 + 10,000 − 8,100 = **11,900**
- IoU = 8,100 ÷ 11,900 ≒ **0.681**

#### Verify by hand
Let us compute the "shifted" case by hand.

- truth: `(100,100)` to `(200,200)` -> area **10,000**
- prediction: `(110,110)` to `(210,210)` -> area **10,000**
- overlap: `(110,110)` to `(200,200)` -> 90 x 90 = **8,100**
- union of the two boxes: 10,000 + 10,000 - 8,100 = **11,900**
- IoU = 8,100 / 11,900 = **0.681**

<!-- ne:begin end -->
"अलि सरेको" अवस्था हातैले गणना गरौं।

- सही: `(100,100)` देखि `(200,200)` → क्षेत्रफल **10,000**
- अनुमान: `(110,110)` देखि `(210,210)` → क्षेत्रफल **10,000**
- प्रतिच्छेद: `(110,110)` देखि `(200,200)` → 90 × 90 = **8,100**
- संयोग: 10,000 + 10,000 − 8,100 = **11,900**
- IoU = 8,100 ÷ 11,900 ≒ **0.681**
<!-- ne:end -->


---

#### 四角を動かして、交差・和集合・IoU を見る
#### Move a box and watch the intersection, the union and IoU

<!-- ne:begin after-heading -->
#### आयत सारेर प्रतिच्छेद, संयोग र IoU हेर्ने
<!-- ne:end -->

`offset_x` と `offset_y` だけを動かします。**予測の四角** が動きます。

Only `offset_x` and `offset_y` change — the **predicted** box moves.

| 予想 / Predict | 見るもの / What to look for |
|---|---|
| ずれを 0 にする / offset 0 | IoU はいくつ？ / what is IoU? |
| ずれを大きくする / increase the offset | 交差と和集合はどうなる？ / what happens to intersection and union? |
| まったく重ならない / no overlap | IoU はいくつ？ / what is IoU? |

ウィジェットが無くても、下のパラメータセルで同じ実験ができます。
The parameter cell below runs the same experiment without a widget.

<!-- ne:begin end -->
`offset_x` र `offset_y` मात्र सारिन्छन्। सर्ने **अनुमानको आयत** हो।

| अनुमान | के हेर्ने |
|---|---|
| सराइ 0 | IoU कति? |
| सराइ बढाउने | प्रतिच्छेद र संयोग के हुन्छ? |
| केही ओभरल्याप नहुने | IoU कति? |

widget नभए पनि तलको parameter सेलले उही प्रयोग गर्न दिन्छ।
<!-- ne:end -->


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import matplotlib.pyplot as plt

truth_box = (60, 60, 110, 90)     # (x, y, 幅, 高さ) / (x, y, width, height)
offset_x = 40                     # ← 動かす値 / change this
offset_y = 20                     # ← 動かす値 / change this
predicted_box = (truth_box[0] + offset_x, truth_box[1] + offset_y, truth_box[2], truth_box[3])


def intersection_and_union(box_a, box_b):
    left = max(box_a[0], box_b[0])
    top = max(box_a[1], box_b[1])
    right = min(box_a[0] + box_a[2], box_b[0] + box_b[2])
    bottom = min(box_a[1] + box_a[3], box_b[1] + box_b[3])
    intersection = max(0, right - left) * max(0, bottom - top)
    union = box_a[2] * box_a[3] + box_b[2] * box_b[3] - intersection
    return intersection, union


intersection, union = intersection_and_union(truth_box, predicted_box)
print(f"交差 / intersection = {intersection} px²")
print(f"和集合 / union      = {union} px²")
print(f"IoU                 = {iou(truth_box, predicted_box):.3f}")

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.set_xlim(0, 300); ax.set_ylim(260, 0); ax.grid(alpha=0.2)
ax.add_patch(plt.Rectangle(truth_box[:2], truth_box[2], truth_box[3], fill=False,
                           edgecolor="#1f6feb", linewidth=2))
ax.add_patch(plt.Rectangle(predicted_box[:2], predicted_box[2], predicted_box[3], fill=False,
                           edgecolor="#9a6700", linewidth=2, linestyle="--"))
ax.set_title(f"IoU = {iou(truth_box, predicted_box):.3f}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    output = widgets.Output()

    def draw(offset_x, offset_y):
        box_a = (60, 60, 110, 90)
        box_b = (box_a[0] + offset_x, box_a[1] + offset_y, box_a[2], box_a[3])
        left, top = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
        right = min(box_a[0] + box_a[2], box_b[0] + box_b[2])
        bottom = min(box_a[1] + box_a[3], box_b[1] + box_b[3])
        intersection = max(0, right - left) * max(0, bottom - top)
        union = box_a[2] * box_a[3] + box_b[2] * box_b[3] - intersection
        fig, ax = plt.subplots(figsize=(5.2, 5.2))
        ax.set_xlim(0, 300); ax.set_ylim(260, 0); ax.grid(alpha=0.2)
        ax.add_patch(plt.Rectangle(box_a[:2], box_a[2], box_a[3], fill=False,
                                   edgecolor="#1f6feb", linewidth=2))
        ax.add_patch(plt.Rectangle(box_b[:2], box_b[2], box_b[3], fill=False,
                                   edgecolor="#9a6700", linewidth=2, linestyle="--"))
        ax.set_title(f"intersection={intersection}  union={union}  "
                     f"IoU={iou(box_a, box_b):.3f}", fontsize=10)
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    widgets.interact(draw,
                     offset_x=widgets.IntSlider(value=40, min=-100, max=140, step=5,
                                                description="offset x"),
                     offset_y=widgets.IntSlider(value=20, min=-100, max=140, step=5,
                                                description="offset y"))
    display(output)

## Part 4　事前学習モデルと重みファイル / Pretrained models and weight files

<!-- ne:begin after-heading -->
## Part 4 पहिले नै तालिम भएको model र weight फाइल
<!-- ne:end -->


#### 「学習済み」を持ってくる

<!-- ne:begin after-heading -->
#### "तालिम भइसकेको" model ल्याउने

सुरुदेखि तालिम गर्न धेरै तस्बिर र लामो गणना समय चाहिन्छ। कति चाहिन्छ भन्ने कुरा model, डेटा र कम्प्युटरअनुसार
धेरै फरक पर्छ। त्यसैले **पहिले नै धेरै तस्बिरमा तालिम भएको model** प्रयोग गरिन्छ। यसलाई
**पहिले नै तालिम भएको model (pretrained model)** भनिन्छ।

**weight फाइल** भनेको त्यो तालिमको नतिजा राखिएको फाइल हो।

| शब्द | अर्थ |
|---|---|
| pretrained model | अरू डेटामा पहिले नै तालिम भएको model |
| weight | तालिमले तय गरेका सङ्ख्याहरूको समूह |
| अनुमान (inference) | तालिम भएको model ले नयाँ तस्बिर जाँच्ने काम |
| fine-tuning | pretrained model लाई आफ्नै डेटामा थप तालिम गर्ने |

दिन ४ मा गरिने काम **fine-tuning** हो। COCO मा सिकेको "वस्तुको आकार हेर्ने क्षमता" उधारो लिएर PVC को
दोष सिकाइन्छ। यो तालिमको अभिलेखमा दिन ४ को fine-tuning (15 epoch, CPU) करिब 90 सेकेन्डमा सकियो
(`training_results/day04/pvc_defect_quick/results.csv` को `time` स्तम्भ, अन्तिम लाइन 89.7 सेकेन्ड),
किनभने यो तालिमको डेटासेट सानो छ। तस्बिर वा epoch बढाउँदा समय पनि बढ्छ।
<!-- ne:end -->
ゼロから学習すると、大量の画像と長い計算時間が必要になります。
どれだけ必要かは、モデル・データ・計算機で大きく変わります。
そこで、**すでに大量の画像で学習済みのモデル** を使います。これを
**事前学習モデル（pretrained model）** と呼びます。

**重みファイル（weight file）** は、その学習結果が入ったファイルです。

| 用語 | 意味 |
|---|---|
| 事前学習モデル | 他のデータで学習済みのモデル |
| 重み（weights） | 学習で決まった数値の集まり |
| 推論（inference） | 学習済みモデルで新しい画像を判定すること |
| ファインチューニング | 事前学習モデルを、自分のデータで追加学習すること |

第 4 日で行うのは **ファインチューニング** です。
COCO で学んだ「物の形を見る力」を借りて、塩ビの欠陥を覚えさせます。
この研修の記録では、第 4 日のファインチューニング（15 エポック、CPU）は約 90 秒で終わりました
（`training_results/day04/pvc_defect_quick/results.csv` の `time` 列、最終行 89.7 秒）。
研修のデータセットが小さいためです。画像やエポックを増やすと、時間も増えます。

#### Starting from something already trained
Training from scratch needs a large image set and a long computation; how much depends on the
model, the data and the hardware. Instead we start from a model **already trained on a large
dataset**: a **pretrained model**.

A **weight file** holds the result of that training.

| Term | Meaning |
|---|---|
| pretrained model | a model already trained on other data |
| weights | the numbers determined by training |
| inference | applying a trained model to new images |
| fine-tuning | further training of a pretrained model on your own data |

Day 4 performs **fine-tuning**: borrow the "shape understanding" learned on COCO and teach it
PVC defects. In this course's recorded run, fine-tuning (15 epochs, CPU) took about 90 seconds
(the `time` column of `training_results/day04/pvc_defect_quick/results.csv`, final row 89.7 s)
because the course dataset is small. More images or epochs mean more time.


#### 任意：チェックポイントと書き出し / Optional: checkpoints and export

<!-- ne:begin after-heading -->
#### वैकल्पिक: checkpoint र export
<!-- ne:end -->

最初の推論後に読む技術補足です。

Read this technical extension after your first inference.

| ファイル | 中身 |
|---|---|
| `.pt`（PyTorch） | **学習のチェックポイント**。モデルの構成情報、学習で決まったパラメータ、学習途中の状態が入っています |
| `.onnx`（ONNX） | **書き出した計算グラフとパラメータ**。学習の状態ではなく、別の実行環境で動かすための形式です |

`.pt` を `.onnx` に書き出すと、学習の記録（エポック数や最適化の状態）は入りません。
「学習を続けられるファイル」と「別の環境で実行するファイル」は別物です。

| File | Contents |
|---|---|
| `.pt` (PyTorch) | a **training checkpoint**: model configuration, the parameters fixed by training, and training state |
| `.onnx` (ONNX) | an **exported computation graph with its parameters**: a format for running the model in another environment, not a training state |

Exporting a `.pt` model to `.onnx` does not carry the training record (epoch count, optimiser
state). A file you can keep training and a file you can run elsewhere are different things.

<!-- ne:begin end -->
पहिलो अनुमानपछि पढ्ने प्राविधिक थप जानकारी।

| फाइल | भित्र के हुन्छ |
|---|---|
| `.pt` (PyTorch) | **तालिमको checkpoint**: model को बनावट, तालिमले तय गरेका parameter र तालिमको अवस्था |
| `.onnx` (ONNX) | **export गरिएको गणना ग्राफ र parameter**: तालिमको अवस्था होइन, अर्को वातावरणमा चलाउन बनाइएको ढाँचा |

`.pt` लाई `.onnx` मा export गर्दा तालिमको अभिलेख (epoch सङ्ख्या, optimizer को अवस्था) सर्दैन।
"तालिम जारी राख्न मिल्ने फाइल" र "अर्को वातावरणमा चलाउने फाइल" फरक कुरा हुन्।
<!-- ne:end -->


#### COCOモデルと塩ビ欠陥クラス / COCO models and PVC defect classes

<!-- ne:begin after-heading -->
#### COCO model र PVC दोष class
<!-- ne:end -->

COCO事前学習モデルには、この教材のblack_spot/scratch/streakという欠陥クラスがありません。
意図した欠陥名を信頼して付けられません。何も出さないとは限らず、無関係なクラスを返すこともあります。
検出0件も「塩ビが正常」の証拠ではありません。第4日に専用ラベルで学習します。

A COCO-pretrained model lacks this course's black_spot/scratch/streak defect classes.
It cannot reliably label the intended defect types. It may still emit an unrelated class;
zero detections do not prove PVC is normal. Day 4 fits a model with the course's defect labels.

<!-- ne:begin end -->
COCO pretrained model सँग यो सामग्रीका `black_spot` / `scratch` / `streak` दोष class छैनन्। त्यसैले
चाहेको दोषको नाम विश्वासका साथ राख्न सकिँदैन। केही नदेखाउने भन्ने पनि छैन; सम्बन्ध नभएको class फर्काउन
सक्छ। पहिचान शून्य हुनु पनि "PVC सामान्य छ" भन्ने प्रमाण होइन। दिन ४ मा यही तालिमका लेबलले model तालिम गरिन्छ।
<!-- ne:end -->


---

## 演習 / Exercises

<!-- ne:begin after-heading -->
## अभ्यास
<!-- ne:end -->


#### 演習 1（基本）：YOLO 形式に変換する

<!-- ne:begin after-heading -->
#### अभ्यास 1 (आधारभूत): YOLO ढाँचामा बदल्ने
<!-- ne:end -->
画像サイズが 640 × 480、ボックスが `x1=320, y1=200, x2=400, y2=260` のとき、
YOLO 形式の 4 つの値を計算してください。

完了の目安：4 つの値がすべて 0〜1 の範囲にあり、小数 6 桁で表示されていること。

#### Exercise 1 (basic): convert to YOLO format
For a 640x480 image and a box `x1=320, y1=200, x2=400, y2=260`, compute the four
YOLO values.
Done when all four values are between 0 and 1 and printed with 6 decimals.

<!-- ne:begin end -->
तस्बिरको आकार 640 × 480 र बक्स `x1=320, y1=200, x2=400, y2=260` हुँदा YOLO ढाँचाका चार मान गणना गर्नुहोस्।

पूरा भएको मापन: चारै मान 0–1 दायराभित्र छन् र 6 दशमलवसम्म देखाइएका छन्।
<!-- ne:end -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
width, height = 640, 480
x1, y1, x2, y2 = 320, 200, 400, 260

xc = 0.0   # <- 計算 / compute
yc = 0.0
w = 0.0
h = 0.0

print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

#### 演習 2（中級）：IoU を手計算して確かめる

<!-- ne:begin after-heading -->
#### अभ्यास 2 (मध्यम): IoU हातैले गणना गरेर जाँच्ने
<!-- ne:end -->
正解 `(50, 50, 150, 150)`、予測 `(100, 50, 200, 150)` の IoU を
**紙で計算** してから、下のセルで確認してください。
重なりの面積が、2 つの枠を合わせた領域の何割になるかを考えます。

完了の目安：紙で計算した値と `iou(...)` の値が一致すること（0.333）。

#### Exercise 2 (intermediate): a hand calculation of IoU
Compute the IoU of truth `(50, 50, 150, 150)` and prediction `(100, 50, 200, 150)`
**on paper**, then check it with the cell. Think about what share of the union of the two boxes
the intersection covers.
Done when your paper result matches `iou(...)` (it is 0.333).

<!-- ne:begin end -->
सही `(50, 50, 150, 150)` र अनुमान `(100, 50, 200, 150)` को IoU **कागजमा** गणना गरेपछि तलको सेलले जाँच्नुहोस्।
प्रतिच्छेदको क्षेत्रफल दुई आयत मिलेर बन्ने क्षेत्रको कति अंश हो, सोच्नुहोस्।

पूरा भएको मापन: कागजको मान र `iou(...)` को मान मिल्नुपर्छ (0.333)।
<!-- ne:end -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
print("紙の計算 / on paper:", "")
print("コードの値 / code   :", round(iou((50, 50, 150, 150), (100, 50, 200, 150)), 3))

#### 演習 3（発展・任意）：第4日で何を学ぶ必要があるかを整理する

<!-- ne:begin after-heading -->
#### अभ्यास 3 (वैकल्पिक): दिन ४ मा के सिकाउनुपर्छ भनेर योजना बनाउने
<!-- ne:end -->
事前学習モデルが塩ビ欠陥を検出できない 4 つの理由を踏まえ、
第 4 日で用意すべきものを整理してください。

| 項目 | あなたの答え |
|---|---|
| クラス名 | |
| 1 枚あたりの欠陥数（目安） | |
| 必要な画像枚数（学習用） | |
| 正解データの形式 | |

完了の目安：表が埋まり、ラベル形式の欄が YOLO の中心形式（`class_id x_center y_center width height`）になっていること。

#### Exercise 3 (optional challenge): plan what Day 4 must teach the model
Given the four reasons, plan what Day 4 must prepare: class names, defects per image,
number of training images, and the ground-truth format.
Done when the table is filled and the label-format field states the YOLO centre format
(`class_id x_center y_center width height`).

<!-- ne:begin end -->
pretrained model ले PVC को दोष किन भेट्न सक्दैन भन्ने चार कारणका आधारमा दिन ४ मा के तयार गर्नुपर्छ भनेर
व्यवस्थित गर्नुहोस्।

| विषय | तपाईंको उत्तर |
|---|---|
| class को नाम | |
| प्रति तस्बिर दोषको सङ्ख्या (अनुमान) | |
| चाहिने तस्बिर सङ्ख्या (प्रशिक्षणका लागि) | |
| सही लेबलको ढाँचा | |

पूरा भएको मापन: तालिका भरिएको छ र लेबल ढाँचाको महलमा YOLO को केन्द्र ढाँचा
(`class_id x_center y_center width height`) लेखिएको छ।
<!-- ne:end -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
plan = {
    "classes": [],
    "defects_per_image": "",
    "training_images": "",
    "label_format": "",
}
for key, value in plan.items():
    print(f"{key:<20} {value if value else '(未記入 / not written)'}")

---

#### 理解度チェック / Understanding questions

<!-- ne:begin after-heading -->
#### बुझाइ जाँच्ने प्रश्न
<!-- ne:end -->

1. YOLO の出力は何ですか。3 つの要素を挙げてください。
2. confidence が 0.9 の検出は、必ず正しいですか。理由も書いてください。
3. IoU が 0.3 のとき、重なりの面積は「何の」30 % ですか。
4. 事前学習モデルが塩ビ欠陥を検出できない理由を 2 つ挙げてください。
5. 「推論」と「学習」の違いを、時間の観点から説明してください。

1. What does YOLO output? Name three components.
2. Is a detection with confidence 0.9 always correct? Explain.
3. When IoU is 0.3, the intersection is 30% of what?
4. Give two reasons a pretrained model cannot detect PVC defects.
5. Explain the difference between inference and training in terms of time.
Model answers: `docs/instructor/day03_instructor_guide.md`.

<!-- ne:begin end -->
1. YOLO को नतिजामा के-के हुन्छ? तीन कुरा भन्नुहोस्।
2. confidence 0.9 भएको पहिचान सधैं सही हुन्छ? कारण पनि लेख्नुहोस्।
3. IoU 0.3 हुँदा प्रतिच्छेद "कसको" 30% हो?
4. pretrained model ले PVC को दोष किन भेट्न सक्दैन, दुई कारण भन्नुहोस्।
5. "अनुमान" र "प्रशिक्षण" बीचको फरक समयको दृष्टिले बताउनुहोस्।
<!-- ne:end -->


---

## 困ったときは / Troubleshooting

<!-- ne:begin after-heading -->
## समस्या परेमा
<!-- ne:end -->

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| YOLO 形式の値が 1 を超える | YOLO values exceed 1 | 画像の幅・高さで割り忘れています。`/ image_width` を確認してください。 | You forgot to divide by the image size. |
| IoU がマイナスになる | IoU is negative | 重なりが無いときは 0 になります。`max(0.0, ...)` を忘れていないか確認してください。 | It should be 0 when there is no overlap; check `max(0.0, ...)`. |
| `confidence` と `確率` を混同してしまう | Confusing confidence with probability | confidence はモデルが出したスコアです。正しい確率ではありません。正しさは正解ラベルと比べて確認します。 | Confidence is a score produced by the model, not a probability of correctness; check correctness against ground truth. |
| 数値がノートの例と少し違う | Numbers differ slightly | 丸めの桁数が違うだけのことが多いです。`round()` の桁を確認してください。 | Usually just a rounding difference; check the digits in `round()`. |

<!-- ne:begin end -->
त्रुटि आएपछि पहिले "कुन सेलमा", "कुन लाइनमा", "के सन्देश आयो" भनेर लेख्नुहोस्। बुझाउनुपर्ने काममा पनि
त्यही अभिलेख राख्नुहोस्।

| लक्षण | के गर्ने |
| --- | --- |
| YOLO ढाँचाको मान 1 भन्दा बढी हुन्छ | तस्बिरको चौडाइ-उचाइले भाग गर्न बिर्सिएको छ। `/ image_width` जाँच्नुहोस्। |
| IoU ऋणात्मक आउँछ | ओभरल्याप नभए 0 हुनुपर्छ। `max(0.0, ...)` लेख्न बिर्सिएको छ कि जाँच्नुहोस्। |
| `confidence` र सम्भावना मिसाउँछु | confidence model ले दिएको स्कोर हो, सही हुने सम्भावना होइन। सही छ कि छैन सही लेबलसँग तुलना गरेर जाँच्नुहोस्। |
| सङ्ख्या नोटको उदाहरणभन्दा अलि फरक आउँछ | प्रायः राउन्ड गरेको दशमलव सङ्ख्या फरक हुन्छ। `round()` मा लेखिएको अङ्क जाँच्नुहोस्। |
<!-- ne:end -->


## 完了チェックリスト / Completion checklist

<!-- ne:begin after-heading -->
## सम्पन्नता जाँचसूची
<!-- ne:end -->

**必須**は下の「必須 / Required」だけです。**演習 3 と追加チャレンジは任意（発展・任意）** で、
未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the items under **必須 / Required** are mandatory. **Exercise 3 and the extra challenge are
optional**; leaving them undone still counts as complete. Do them only if you have spare time.

**必須 / Required**

- [ ] Part 1〜Part 4 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-4 and ran every code cell
- [ ] YOLO 形式の座標を自分で計算できた
      - [ ] Computed YOLO coordinates yourself
- [ ] IoU を手計算し、コードと一致した
      - [ ] Hand-computed IoU and matched the code
- [ ] 演習 1・演習 2（必須）を実施した
      - [ ] Completed Exercises 1 and 2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 understanding questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

**任意 / Optional**

- [ ] 演習 3（発展・任意）：第 4 日の準備項目を整理した
      - [ ] Exercise 3 (optional challenge): planned what Day 4 must prepare
- [ ] 追加チャレンジ：COCO の 80 クラスから製造検査に使えそうなものを探した
      - [ ] Extra challenge: looked for COCO classes useful for inspection

<!-- ne:begin end -->
**अनिवार्य** भनेको तलको "अनिवार्य" सूची मात्र हो। **अभ्यास 3 र थप चुनौती वैकल्पिक** हुन्; नगरे पनि पूरा
मानिन्छ। समय बाँकी भएमा मात्र गर्नुहोस्।

**अनिवार्य**

- [ ] Part 1–Part 4 पढेर सबै code सेल चलाएँ
- [ ] YOLO ढाँचाको निर्देशाङ्क आफैं गणना गरें
- [ ] IoU हातैले गणना गरेर कोडसँग मिलाएँ
- [ ] अभ्यास 1 र 2 (अनिवार्य) गरें
- [ ] बुझाइ जाँच्ने 5 प्रश्नको उत्तर दिएँ
- [ ] नोटबुक सुरक्षित गरें

**वैकल्पिक**

- [ ] अभ्यास 3 (वैकल्पिक): दिन ४ का तयारीका बुँदा व्यवस्थित गरें
- [ ] थप चुनौती: COCO का 80 class मध्ये उत्पादन निरीक्षणमा काम लाग्ने कुनै छ कि हेरें
<!-- ne:end -->


## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

<!-- ne:begin after-heading -->
## थप अध्ययन (वैकल्पिक)
<!-- ne:end -->

- [Ultralytics ドキュメント](https://docs.ultralytics.com/)
- [Ultralytics: 物体検出タスク](https://docs.ultralytics.com/tasks/detect/)
- [COCO データセットのクラス一覧](https://cocodataset.org/#explore)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

COCO の 80 クラスの一覧を見て、製造検査に使えそうなクラスがあるか探してください。

Look at the 80 COCO classes and see whether any of them could be useful for inspection.

<!-- ne:begin end -->
### सन्दर्भ लिङ्क

- [Ultralytics कागजात](https://docs.ultralytics.com/)
- [Ultralytics: वस्तु पहिचान कार्य](https://docs.ultralytics.com/tasks/detect/)
- [COCO डेटासेटका class को सूची](https://cocodataset.org/#explore)

### थप चुनौती (अनिवार्य होइन)

COCO का 80 class को सूची हेरेर उत्पादन निरीक्षणमा काम लाग्ने class छ कि छैन खोज्नुहोस्।
<!-- ne:end -->
